# 03 · Demo and merged-adapter latency check

1. **Demo**: a Gradio page (paste an email → validated JSON) with a temporary public `gradio.live` link.
2. **Latency check (optional)**: does merging the LoRA adapter into the 4-bit base fix the fine-tuned model's slowdown (13.4 s vs 8.4 s per email), and does it change accuracy?

Runtime: **GPU T4**. The adapter is downloaded from the Hugging Face Hub
([BenarbiaEmir/qwen3-4b-recruitment-email-json-qlora](https://huggingface.co/BenarbiaEmir/qwen3-4b-recruitment-email-json-qlora)); no Drive needed for the demo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Benarbiaa/email-extraction-qlora.git"
import os
if not os.path.exists("/content/email-extraction-qlora"):
    !git clone -q {REPO_URL} /content/email-extraction-qlora
%cd /content/email-extraction-qlora
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets jsonschema gradio
import transformers, peft, gradio, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| gradio", gradio.__version__,
      "| torch", torch.__version__)

In [ ]:
# Optional: Hugging Face token from Colab Secrets (key icon, name HF_TOKEN, notebook access on).
# Faster model downloads; needed later to push the adapter. Skipped if the secret is missing.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set from Colab secrets")
except Exception as e:
    print("no HF_TOKEN secret, downloading anonymously:", type(e).__name__)


In [ ]:
ADAPTER = "BenarbiaEmir/qwen3-4b-recruitment-email-json-qlora"   # or a local dir, e.g. a Drive path
GOLD = "data/splits/test.jsonl" if os.path.exists("data/splits/test.jsonl") else "data/splits/test_provisional.jsonl"

## 2. Demo
Wait for the `Running on public URL: https://....gradio.live` line and open it (valid ~72 h, or until the cell stops).
Three example emails are built in. **Stop this cell** (■) to continue with section 3.
Set `MERGE = True` to serve the merged adapter instead (faster if section 3 confirms it).

In [ ]:
MERGE = False
!python app/app.py --adapter {ADAPTER} --share {"--merge" if MERGE else ""}

## 3. Optional: merged vs unmerged adapter
**3a. Quick speed check** (~10 min): the same 20 test emails, unmerged then merged.

In [ ]:
!rm -f /tmp/unmerged.jsonl /tmp/merged.jsonl
!python src/infer.py --mode finetuned --adapter {ADAPTER} --split {GOLD} --limit 20 --out /tmp/unmerged.jsonl
!python src/infer.py --mode finetuned --adapter {ADAPTER} --merge --split {GOLD} --limit 20 --out /tmp/merged.jsonl
import json, statistics
for name in ["unmerged", "merged"]:
    rows = [json.loads(l) for l in open(f"/tmp/{name}.jsonl")]
    print(f"{name:9} mean latency {statistics.mean(r['latency_s'] for r in rows):.2f} s")
same = sum(a["output"] == b["output"] for a, b in zip(map(json.loads, open("/tmp/unmerged.jsonl")),
                                                      map(json.loads, open("/tmp/merged.jsonl"))))
print(f"identical outputs: {same}/20")

**3b. Full test run with the merged adapter** (~30–45 min). Only worth it if 3a shows a clear speed-up.
Scored as a separate row (`ft_qlora_v1_merged`): merging re-quantizes the weights, so accuracy must be re-measured, not assumed.
Predictions and metrics are saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/email-extraction-qlora"
!mkdir -p {DRIVE_DIR}/preds {DRIVE_DIR}/tables
!rm -rf results/preds && mkdir -p results && ln -s {DRIVE_DIR}/preds results/preds
!python src/infer.py --mode finetuned --adapter {ADAPTER} --merge --split {GOLD} --out results/preds/ft_qlora_v1_merged.jsonl
!python src/eval.py --pred results/preds/ft_qlora_v1_merged.jsonl --gold {GOLD} --name ft_qlora_v1_merged
!cp results/tables/summary.csv results/tables/metrics_ft_qlora_v1_merged.json {DRIVE_DIR}/tables/